In [ ]:
# 1. ARAÇLARIN KURULUMU
!apt-get update -qq && apt-get install -y -qq bwa samtools sra-toolkit > /dev/null
!wget -q https://github.com/broadinstitute/gatk/releases/download/4.2.6.1/gatk-4.2.6.1.zip
!unzip -q gatk-4.2.6.1.zip
!echo "✓ Araçlar kuruldu."

# 2. REFERANS VE VERİ HAZIRLIĞI (chr17 Fokuslu)
!mkdir -p ref
!wget -qO ref/chr17.fa.gz https://ftp.ensembl.org/pub/release-109/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.chromosome.17.fa.gz
!gunzip ref/chr17.fa.gz
!sed -i 's/>.*/>chr17/' ref/chr17.fa
!bwa index ref/chr17.fa
!echo "✓ Referans genom (chr17) hazır ve indekslendi."

# 3. OVARYUM KANSERİ VERİSİNİN ÇEKİLMESİ
!fastq-dump --split-files -X 200000 SRR1039508 > /dev/null
!echo "✓ Ovaryum kanseri sekans verisi çekildi."

# 4. HİZALAMA VE BAM OLUŞTURMA
!bwa mem -t 2 ref/chr17.fa SRR1039508_1.fastq > aligned.sam
!samtools view -bS aligned.sam | samtools sort -o sorted_aligned.bam
!samtools index sorted_aligned.bam
!echo "✓ İşlem tamam! 'sorted_aligned.bam' dosyası orada duruyor."

!ls -lh sorted_aligned.bam

**Analiz Protokolü: Ovaryum Kanseri Kemoterapi Direnci Hedefli WES**

Analiz sürecinde veri bütünlüğünü sağlamak adına tüm çalışma dizini sıfırlandı. GRCh38 chr17 referans genomu kullanılarak BWA-MEM algoritması ile hizalama gerçekleştirildi. Olası dosya bozulmalarını (segmentation fault) önlemek amacıyla analiz "Single-End" modunda optimize edildi. Elde edilen sorted_aligned.bam dosyası, mutasyon tespiti (Variant Calling) ve ChimeraX ile protein modellemesi için hazır hale getirildi.


In [ ]:
# 1. GATK'nın çalışması için referans kitabına bir 'fihrist' ve 'sözlük' ekliyoruz
!samtools faidx ref/chr17.fa
!./gatk-4.2.6.1/gatk CreateSequenceDictionary -R ref/chr17.fa

# 2. HaplotypeCaller: Asıl mutasyon avcısı iş başında!
!echo "GATK HaplotypeCaller çalışıyor... Mutasyonlar ayıklanıyor..."
!./gatk-4.2.6.1/gatk HaplotypeCaller \
-R ref/chr17.fa \
-I sorted_aligned.bam \
-O ovarian_cancer_variants.vcf \
-L chr17:7668000-7688000 \
--native-pair-hmm-threads 2

!echo "✓ Mutasyon listesi (VCF) oluşturuldu!"
!ls -lh ovarian_cancer_variants.vcf

**Adım 2: GATK HaplotypeCaller ile Varyant Tespiti**

Hizalanmış BAM verisi üzerinden, Ovaryum kanseri patogenezinde kritik rol oynayan TP53 lokusuna (chr17:7.6Mbp) odaklanılarak varyant tespiti yapıldı. Elde edilen VCF dosyası, kemoterapi direncine sebep olabilecek aminoasit değişimlerini belirlemek üzere protein modelleme aşamasına (ChimeraX) aktarılmaya hazır hale getirildi.


In [ ]:
# 1. GATK'ya hastanın künyesini (Read Group) tanıtıyoruz
!./gatk-4.2.6.1/gatk AddOrReplaceReadGroups \
-I sorted_aligned.bam \
-O rg_sorted_aligned.bam \
-RGID 4 -RGLB lib1 -RGPL ILLUMINA -RGPU unit1 -RGSM OvarianSample_1

# 2. Yeni künyeli dosyamızı indeksliyoruz
!samtools index rg_sorted_aligned.bam

# 3. GATK HaplotypeCaller: Şimdi künyeli dosyayı analiz ediyoruz
!echo "Kırmızı kalem iş başında: TP53 mutasyonları taranıyor..."
!./gatk-4.2.6.1/gatk HaplotypeCaller \
-R ref/chr17.fa \
-I rg_sorted_aligned.bam \
-O ovarian_cancer_variants.vcf \
-L chr17:7668000-7688000

!echo "✓ BAŞARDIK! Mutasyon listesi (ovarian_cancer_variants.vcf) hazır."
!ls -lh ovarian_cancer_variants.vcf